# 8.9 - Multimodal RAG

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook works through Multimodal RAG hands-on: Describe an image, then embed the description; Extract a table to markdown, then embed it; Unify text, tables, and images in one index; Direct vision RAG: retrieve on descriptions, generate with the real image; ColPali / ColQwen: vision-first retrieval with no OCR; Document AI: layout-aware parsing with Unstructured.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install the multimodal stack

One line that installs every library the lesson touches: the Gemini SDK for vision and embeddings, ColPali via byaldi, Unstructured for document parsing, OpenCLIP for joint embeddings, faster-whisper for audio, and Google Cloud Vision for OCR. It is commented out so it only runs when you uncomment it on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy byaldi "unstructured[pdf]" open_clip_torch torch faster-whisper google-cloud-vision google-genai python-dotenv -q  # noqa


**Explanation**

Pure environment prep - no code runs here until you uncomment it. It gathers the full toolset for all ten steps in one place so later cells import cleanly.

**How the code works - step by step**
- **numpy** - vector math for the by-hand cosine/dot-product retrieval.
- **byaldi** - the reference wrapper for ColPali/ColQwen vision-first retrieval (step 5).
- **unstructured[pdf]** - layout-aware document parsing (step 6).
- **open_clip_torch, torch** - joint image-text embeddings (step 7).
- **faster-whisper** - fast timestamped audio transcription (step 8).
- **google-cloud-vision** - bilingual Devanagari/English OCR (step 10).
- **google-genai** - the current Gemini SDK used for vision description and text embeddings throughout.
- **python-dotenv** - loads keys from a .env file.

**In one line:** one pip line stages every library the ten steps need.

**What you'll see:** (no output - environment prep; the line is commented so nothing runs until you uncomment it)

## Load your API keys from the environment

Before any Gemini or Google Vision call, this pulls credentials from the environment rather than hardcoding them. Any one provider key is enough to start the by-hand steps.

> **Needs a Google AI Studio key** (set GOOGLE_API_KEY from aistudio.google.com); the OCR step also uses GOOGLE_APPLICATION_CREDENTIALS.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_APPLICATION_CREDENTIALS", "") # 
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A tiny credentials-hygiene cell, not a model call. It seeds two environment variables to empty defaults so downstream code can read them without a KeyError, and reminds you to supply real values yourself.

**How the code works - step by step**
- **`os.environ.setdefault("GOOGLE_APPLICATION_CREDENTIALS", "")`** - reserves the service-account path used by Google Cloud Vision in step 10.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - reserves the AI Studio key that `genai.Client()` reads automatically.
- Both use `setdefault`, so a key you already exported in your shell is left untouched.

**In one line:** read keys from the environment, never hardcode them.

**What you'll see:** (no output - it only sets environment variables)

## 1 - Describe an image, then embed the description

This is the single move the whole lesson is built on: retrieval can only rank things that live in its vector space, and that space is text. So you have a vision model DESCRIBE a chart into a rich sentence, embed that sentence, and now a chart is as findable as a paragraph.

> **Needs a Google AI Studio key** (set GOOGLE_API_KEY).

In [ ]:
# DESCRIBE AN IMAGE AS SEARCHABLE TEXT - vision to text embedding
# pip install google-genai pillow numpy
from google import genai
from PIL import Image, ImageDraw

client = genai.Client()   # reads GOOGLE_API_KEY from the environment

def describe_image_for_rag(image, context=""):
    prompt = ("Describe this image for a search index. State what kind of visual it is, "
              "every number, label, and trend, and what someone might search for.")
    if context:
        prompt += f"\nDocument context: {context}"
    # google-genai takes a PIL image directly in contents (auto-converted to a part)
    return client.models.generate_content(
        model="gemini-2.5-flash", contents=[prompt, image]).text.strip()

# a synthetic revenue bar chart
img = Image.new("RGB", (400, 250), "white"); d = ImageDraw.Draw(img)
d.text((20, 10), "Netsetos Revenue (Lakhs)", fill="black")
for i, (q, v) in enumerate([("Q1",45),("Q2",72),("Q3",98),("Q4",135)]):
    h = int(v * 1.2)
    d.rectangle([40+i*85, 220-h, 40+i*85+60, 220], fill="#0d9488")
    d.text((55+i*85, 225), q, fill="black"); d.text((50+i*85, 220-h-15), str(v), fill="black")

desc = describe_image_for_rag(img, context="Netsetos annual report 2024")
print("Description:", desc[:160], "...\n")

# embed the DESCRIPTION (not the pixels) with the text embedder
emb = client.models.embed_content(model="gemini-embedding-001", contents=desc).embeddings[0].values
print(f"Embedded description -> {len(emb)}-dim text vector; searchable by 'Q4 revenue', 'growth trend'.")

# Output:
# Description: A bar chart titled "Netsetos Revenue (Lakhs)" with four bars - Q1: 45, Q2: 72,
# Q3: 98, Q4: 135 - showing steady quarter-over-quarter growth ...
# Embedded description -> 3072-dim text vector; searchable by 'Q4 revenue', 'growth trend'.

**Explanation**

The cell converts pixels into a searchable text vector in two hops - Gemini writes a description, then the text embedder embeds the description (not the image). It is the describe-then-embed pattern in its smallest form, and the prompt is doing the real work: it explicitly asks for every number, label, and trend so the description stays searchable.

**How the code works - step by step**
- **`describe_image_for_rag`** - sends the PIL image plus a SPECIFIC prompt (numbers, labels, trends, likely searches) to `gemini-2.5-flash`; google-genai auto-converts the PIL image into a content part.
- **synthetic chart** - draws a four-bar revenue chart (Q1 45 -> Q4 135) with PIL so there is a real image to describe.
- **`embed_content`** - embeds the returned DESCRIPTION with `gemini-embedding-001`, landing it in the same space as your text chunks.

**In one line:** describe the pixels to text, embed the text.

**What you'll see:** Prints the first 160 characters of Gemini's description (the four quarterly values and 'steady growth'), then confirms a 3072-dim text vector now searchable by queries like 'Q4 revenue' or 'growth trend'.

## 2 - Extract a table to markdown, then embed it

Tables are the modality naive text RAG mangles worst - flattened, a clean grid becomes a soup of numbers with no rows or columns. The fix mirrors step 1: convert the table to STRUCTURED text (markdown) that keeps each price next to its course, then embed that as one chunk.

> **Needs a Google AI Studio key** (set GOOGLE_API_KEY).

In [ ]:
# EXTRACT A TABLE TO MARKDOWN, THEN EMBED AS TEXT
# pip install google-genai pillow
from google import genai
from PIL import Image, ImageDraw

client = genai.Client()

# a synthetic table image
img = Image.new("RGB", (420, 180), "white"); d = ImageDraw.Draw(img)
for i, row in enumerate(["Course       | Price   | Hours | Rating",
                        "GenAI        | 14,999  | 146   | 4.8",
                        "Data Science | 12,999  | 120   | 4.7",
                        "Cloud GCP    | 11,999  | 110   | 4.6"]):
    d.text((15, 15+i*35), row, fill="black")

md = client.models.generate_content(model="gemini-2.5-flash",
    contents=["Extract this table as GitHub markdown. Include every row and column exactly.", img]).text.strip()
print(md, "\n")

# embed the whole markdown table as ONE chunk (keeps rows and columns together)
emb = client.models.embed_content(model="gemini-embedding-001", contents=md).embeddings[0].values
print(f"Embedded table as one {len(emb)}-dim chunk; searchable: 'cheapest course', 'highest rated'.")

# Output:
# | Course | Price | Hours | Rating |
# |---|---|---|---|
# | GenAI | 14,999 | 146 | 4.8 |
# | Data Science | 12,999 | 120 | 4.7 |
# | Cloud GCP | 11,999 | 110 | 4.6 |
# Embedded table as one 3072-dim chunk; searchable: 'cheapest course', 'highest rated'.

**Explanation**

Same describe-then-embed shape as step 1, but the extractor targets structure instead of prose: Gemini is asked to return GitHub markdown with every row and column, and the whole table is embedded as a single chunk so its rows and columns stay together.

**How the code works - step by step**
- **synthetic table image** - draws a four-row pricing grid (Course / Price / Hours / Rating) with PIL.
- **`generate_content`** - sends the image with a prompt to extract it as GitHub markdown, every row and column exactly.
- **`embed_content`** - embeds the returned markdown as ONE chunk (not cell-by-cell), so 'cheapest course' retrieves the intact table.

**In one line:** turn the grid into markdown, embed the whole thing as one chunk.

**What you'll see:** Prints the extracted markdown table (header, separator, and the three course rows), then confirms it was embedded as one 3072-dim chunk searchable by 'cheapest course' or 'highest rated'.

## 3 - Unify text, tables, and images in one index

Steps 1 and 2 produced text from images and tables; now drop all three into ONE index. The payoff is that a query never needs to know where the answer lives - table, chart, or paragraph - because everything is a text vector and a single retriever ranks them together.

In [ ]:
# UNIFIED MULTIMODAL RAG - text + tables + images in ONE index
# pip install google-genai pillow numpy
import numpy as np
from google import genai

client = genai.Client()

class MultimodalRAG:
    def __init__(self):
        self.entries = []   # each: {type, text, emb, source}

    def _embed(self, text):
        return np.array(client.models.embed_content(
            model="gemini-embedding-001", contents=text).embeddings[0].values)

    def add_text(self, text, source="doc"):
        self.entries.append({"type":"text", "text":text, "emb":self._embed(text), "source":source})

    def add_image(self, image, source="img", context=""):
        desc = client.models.generate_content(model="gemini-2.5-flash",
            contents=[f"Describe this image for search. Context: {context}", image]).text.strip()
        self.entries.append({"type":"image", "text":desc, "emb":self._embed(desc), "source":source})

    def add_table(self, markdown, source="table"):
        self.entries.append({"type":"table", "text":markdown, "emb":self._embed(markdown), "source":source})

    def query(self, question, k=3):
        qe = self._embed(question)
        order = sorted(range(len(self.entries)),
                       key=lambda i: -float(np.dot(qe, self.entries[i]["emb"])))
        top = [self.entries[i] for i in order[:k]]
        ctx = "\n\n".join(f"[{e['type'].upper()} {e['source']}]\n{e['text']}" for e in top)
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Answer from context only.\n\n{ctx}\n\nQ: {question}\nA:").text.strip()
        return {"answer": ans, "sources": [(e["type"], e["source"]) for e in top]}

rag = MultimodalRAG()
rag.add_text("Refund policy: full within 7 days, 50% for 8-30 days, none after 30.", "refund.txt")
rag.add_table("| Course | Price |\n|---|---|\n| GenAI | 14,999 |\n| Cloud GCP | 11,999 |", "pricing")
from PIL import Image, ImageDraw
chart = Image.new("RGB", (300,200), "white"); dd = ImageDraw.Draw(chart)
dd.text((10,5), "Students: 2024=2000, 2025=5000", fill="black")
rag.add_image(chart, "growth.png", context="annual report")

for q in ["Which course is cheapest?", "How many students in 2025?", "What is the refund policy?"]:
    r = rag.query(q)
    print(f"Q: {q}\nA: {r['answer'][:80]}  sources={r['sources']}\n")

# Output:
# Q: Which course is cheapest?
# A: Cloud GCP at 11,999.  sources=[('table', 'pricing'), ...]
# Q: How many students in 2025?
# A: 5,000 students.  sources=[('image', 'growth.png'), ...]

**Explanation**

This is the pattern assembled into a small class. Every `add_*` method funnels its input to the same place - a text string plus its embedding tagged with a `type` - so text, image-descriptions, and table-markdown share one store; `query` embeds the question, ranks all entries by dot product, and hands the top mix to Gemini to answer from context only.

**How the code works - step by step**
- **`__init__` / `_embed`** - holds one `entries` list and a helper that embeds any text with `gemini-embedding-001`.
- **`add_text`** - stores raw text directly.
- **`add_image`** - describes the image with Gemini first, then stores that description.
- **`add_table`** - stores markdown as-is.
- **`query`** - embeds the question, sorts every entry by `np.dot` similarity, takes the top-k across all modalities, and asks Gemini to answer from that mixed context.
- **demo** - loads a refund paragraph, a pricing table, and a students chart, then fires three questions.

**In one line:** three add-methods, one shared vector store, one retriever for all modalities.

**What you'll see:** Prints each question with its answer and sources: 'Cheapest course?' -> Cloud GCP 11,999 (from the table), 'students in 2025?' -> 5,000 (from the image description), 'refund policy?' -> from the text - each pulling the right modality.

## 4 - Direct vision RAG: retrieve on descriptions, generate with the real image

Descriptions are lossy, so a precise question ('the exact Q3 bar value?') can fail even when retrieval found the right chart. The fix is a two-speed design: search cheaply on text descriptions (Strategy A), then at generation time send the ACTUAL image to Gemini (Strategy B) so it reads the real pixels.

> **Needs a Google AI Studio key** (set GOOGLE_API_KEY).

In [ ]:
# DIRECT VISION RAG - retrieve on descriptions, generate with the real images
# pip install google-genai pillow numpy
import numpy as np
from google import genai

client = genai.Client()

class VisionRAG:
    def __init__(self):
        self.entries = []

    def _embed(self, text):
        return np.array(client.models.embed_content(
            model="gemini-embedding-001", contents=text).embeddings[0].values)

    def add(self, desc, source, image=None):
        self.entries.append({"desc":desc, "emb":self._embed(desc), "source":source, "image":image})

    def query(self, question, k=2):
        qe = self._embed(question)
        order = sorted(range(len(self.entries)),
                       key=lambda i: -float(np.dot(qe, self.entries[i]["emb"])))[:k]
        parts = [f"Answer from this context only.\nQuestion: {question}\nContext:"]
        for i in order:
            e = self.entries[i]
            parts.append(f"\n[{e['source']}] {e['desc']}")
            if e["image"] is not None:
                parts.append(e["image"])              # send the ACTUAL image (Strategy B)
        ans = client.models.generate_content(model="gemini-2.5-flash", contents=parts).text.strip()
        return {"answer": ans, "sources": [self.entries[i]["source"] for i in order]}

print("Strategy A (retrieve): image -> describe -> embed description -> text search (fast, cheap)")
print("Strategy B (generate): send the ACTUAL image to Gemini so it reads exact chart values")
print("Best practice: A for retrieval, B only when the query needs precise visual detail")

# Output:
# Strategy A (retrieve): image -> describe -> embed description -> text search (fast, cheap)
# Strategy B (generate): send the ACTUAL image to Gemini so it reads exact chart values
# Best practice: A for retrieval, B only when the query needs precise visual detail

**Explanation**

A variant of the step-3 class that keeps the original image alongside its description. Retrieval still runs on the cheap text vectors, but the winning entries' real images are appended to the generation prompt - google-genai accepts a mixed list of strings and PIL images in `contents`, which is the entire multimodal-prompt trick.

**How the code works - step by step**
- **`add`** - stores a description, its embedding, a source label, and the original PIL image (optional).
- **`query`** - embeds the question, ranks by `np.dot`, then builds a `parts` list: the question text plus, for each hit, its description AND its actual image when present.
- **`generate_content`** - passes that mixed text+image list to Gemini so it can read exact values off the pixels.
- **print block** - this cell defines the class and prints the A-vs-B strategy summary rather than running a live query.

**In one line:** retrieve on text (A), generate on the real image (B), route B only when precision is needed.

**What you'll see:** Prints three summary lines: Strategy A retrieves via description (fast, cheap), Strategy B sends the actual image so Gemini reads exact values, and best practice is A for retrieval, B only when the query needs precise visual detail.

## 5 - ColPali / ColQwen: vision-first retrieval with no OCR

Everything so far flattened visuals to text at ingestion. ColPali (ICLR 2025) asks: what if you never OCR? It screenshots each page, encodes it into ~1,024 patch vectors, and matches query tokens directly against page patches via MaxSim - strong on dense, layout-heavy documents.

> **Needs a GPU and model download** - shown for reference; loads a multi-GB ColQwen checkpoint.

In [ ]:
# COLPALI / COLQWEN - vision-first retrieval, no OCR pipeline
# pip install byaldi
from byaldi import RAGMultiModalModel

# load a ColPali-family checkpoint (ColQwen2 v1.0; ColPali-family checkpoints can overfit their training set - validate on YOUR docs)
rag = RAGMultiModalModel.from_pretrained("vidore/colqwen2-v1.0")

# index screenshots each page into ~1,024 patch vectors - no OCR, no chunking
rag.index(input_path="annual_report.pdf", index_name="report", overwrite=True)

# MaxSim late interaction: each query token matches its best page patch, scores summed
results = rag.search("What was Q4 revenue?", k=3)
for r in results:
    print(f"page {r.doc_id}:{r.page_num}  score {r.score:.2f}")

# then pass the winning PAGE IMAGE(s) to a VLM to actually read the answer
# (ColPali finds the page; a vision model reads it - the hybrid pattern)

# Output:
# page 0:12  score 18.42   <- the revenue chart page, found without any OCR
# page 0:3   score 12.07
# page 0:8   score 10.95

**Explanation**

A reference walkthrough of the byaldi wrapper, not a call to Gemini. Three lines cover the whole loop - load a ColPali-family checkpoint, index a PDF by screenshotting each page into patch embeddings, and search with MaxSim late interaction that returns page numbers and scores.

**How the code works - step by step**
- **`from_pretrained("vidore/colqwen2-v1.0")`** - loads a ColQwen checkpoint (the comment warns these can overfit their training set, so validate on your own docs).
- **`index`** - screenshots each PDF page into ~1,024 patch vectors; no OCR, no chunking.
- **`search`** - runs MaxSim (each query token matches its best page patch, scores summed) and returns page ids + scores.
- comment - you then pass the winning page IMAGES to a VLM to read the answer (the retrieve-then-read hybrid).

**In one line:** screenshot pages into patch vectors, match query tokens with MaxSim, no OCR anywhere.

**What you'll see:** Prints the top page hits with MaxSim scores (e.g. page 0:12 score 18.42 - the revenue-chart page found without any OCR). Note the scores are summed per-token maxima, not 0-1 cosines.

## 6 - Document AI: layout-aware parsing with Unstructured

Your by-hand prompts work on clean synthetic images; real PDFs are messy - multi-column, merged cells, scanned skew. Document AI tools detect the layout (where are the tables, titles, figures?) then route each element to the right extractor, turning a raw PDF into the clean text and markdown steps 1-3 assume.

> **Needs the unstructured[pdf] extras** and a real PDF file; hi_res runs a layout-detection model.

In [ ]:
# DOCUMENT AI - layout-aware parsing that routes each element to the right extractor
# pip install "unstructured[pdf]"
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="report.pdf",
    strategy="hi_res",                 # YOLOX/Detectron2 layout detection
    infer_table_structure=True,        # tables come out as HTML, not scrambled text
)

for el in elements[:4]:
    print(el.category, "|", str(el)[:55])

# route by category: Table -> embed el.metadata.text_as_html; NarrativeText -> chunk.
# (figures come as their own Image elements only with extract_image_block_types=["Image"])
tables = [el for el in elements if el.category == "Table"]
print(f"\nfound {len(tables)} table(s); HTML is in el.metadata.text_as_html")

# Output:
# Title | Netsetos Annual Report 2024
# NarrativeText | Revenue grew across all four quarters ...
# Table | Course Price Hours Rating GenAI 14,999 146 4.8 Data Science ...
# NarrativeText | The table above summarizes all four courses ...
# found 1 table(s); HTML is in el.metadata.text_as_html

**Explanation**

A reference snippet showing the partition-then-route pattern. `partition_pdf` with the hi_res strategy runs layout detection and returns a list of typed elements, each carrying a `category`, so you can send tables, figures, and narrative text to their optimal downstream handler.

**How the code works - step by step**
- **`partition_pdf(strategy="hi_res", infer_table_structure=True)`** - runs YOLOX/Detectron2 layout detection and extracts tables as HTML rather than scrambled text.
- **loop over `elements[:4]`** - prints each element's `category` and a text preview.
- **filter by `category == "Table"`** - collects table elements, whose structure lives in `el.metadata.text_as_html`.
- comment - figures only appear as Image elements when you pass `extract_image_block_types`.

**In one line:** detect the layout, tag each element, route tables/figures/text separately.

**What you'll see:** Prints the first four elements with their categories (Title, NarrativeText, Table, NarrativeText) and a text preview, then reports it found 1 table whose HTML is in el.metadata.text_as_html.

## 7 - Multimodal embeddings: image and text in one space

Text grounding describes images to text before embedding. Multimodal embeddings skip the caption: they put images AND text into one shared vector space directly, so a text query retrieves an image by cosine similarity with no VLM in between - faster and cheaper for large image libraries.

> **Needs a model download** - loads OpenCLIP ViT-B-32 weights (~600MB).

In [ ]:
# MULTIMODAL EMBEDDINGS - image and text in ONE space (cross-modal search)
# pip install open_clip_torch pillow torch
import open_clip, torch
from PIL import Image

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k")
tokenizer = open_clip.get_tokenizer("ViT-B-32")

def embed_image(path):
    x = preprocess(Image.open(path)).unsqueeze(0)
    with torch.no_grad(): return model.encode_image(x)[0]

def embed_text(text):
    with torch.no_grad(): return model.encode_text(tokenizer([text]))[0]

# a TEXT query retrieves an IMAGE - both live in the same space
q = embed_text("a bar chart of quarterly revenue")
img = embed_image("chart.png")
sim = torch.cosine_similarity(q, img, dim=0).item()
print(f"text <-> image cosine similarity: {sim:.3f}")
print("High similarity => this image is retrievable by that text query, no caption needed.")

# Output:
# text <-> image cosine similarity: 0.31
# High similarity => this image is retrievable by that text query, no caption needed.

**Explanation**

A minimal cross-modal-search demo with OpenCLIP. Two encoders map a picture and a phrase into the SAME vector space, and because they share that space a text query and an image can be compared directly by cosine similarity - no description step at all.

**How the code works - step by step**
- **`create_model_and_transforms("ViT-B-32", ...)`** - loads the CLIP model, its image preprocessor, and tokenizer.
- **`embed_image`** - preprocesses an image file and runs `encode_image` to get its vector.
- **`embed_text`** - tokenizes a phrase and runs `encode_text` to get its vector in the same space.
- **comparison** - embeds a text query and a chart image, then `cosine_similarity` scores how retrievable that image is by that phrase.

**In one line:** encode image and text into one space, compare by cosine - no caption needed.

**What you'll see:** Prints the text-to-image cosine similarity (e.g. 0.31) and notes that a high value means the image is retrievable by that text query with no caption step.

## 8 - Audio & video RAG: transcribe to timestamped text

Audio and video feel like a new problem but collapse to the same describe-then-embed trick, with one addition: timestamps. You transcribe speech to timestamped segments, embed the text, and keep the timestamp - so retrieval returns not just an answer but the exact second to jump to.

> **Needs a faster-whisper model download**; runs int8 on CPU for the demo.

In [ ]:
# AUDIO RAG - transcribe to TIMESTAMPED chunks, then standard text RAG
# pip install faster-whisper
from faster_whisper import WhisperModel

model = WhisperModel("large-v3-turbo", device="cpu", compute_type="int8")
segments, info = model.transcribe("lecture.mp3")   # segments already carry .start/.end/.text

chunks = []
for seg in segments:
    chunks.append({"start": seg.start, "end": seg.end, "text": seg.text.strip()})
    print(f"[{seg.start:6.1f}s] {seg.text.strip()[:52]}")

# embed each chunk['text'] with your text embedder; KEEP the timestamp so
# retrieval returns the answer AND the second to jump to. Video adds two more
# streams (key-frame VLM descriptions + on-screen OCR) aligned by the same timestamp.

# Output:
# [   0.0s] Welcome to lesson eight point nine on multimodal RAG
# [   6.4s] Real documents are not just text - they have charts
# [  12.1s] Today we build a pipeline over images, tables, and audio

**Explanation**

A standard-text-RAG-over-audio snippet. faster-whisper transcribes an audio file into segments that already carry start/end times, and you build chunks of text plus timestamp - the timestamp is the one piece of metadata that lets retrieval point to a position, not just answer.

**How the code works - step by step**
- **`WhisperModel("large-v3-turbo", device="cpu", compute_type="int8")`** - loads the fast turbo model quantized for a CPU demo.
- **`transcribe`** - returns segments each carrying `.start`, `.end`, and `.text` (no word_timestamps needed).
- **loop** - builds a `{start, end, text}` chunk per segment and prints each with its timestamp.
- comment - embed each chunk's text but KEEP the timestamp; video adds two more time-aligned streams (key-frame VLM descriptions + on-screen OCR).

**In one line:** transcribe to timestamped chunks, embed the text, keep the second.

**What you'll see:** Prints each transcribed segment with its start time (e.g. '[ 0.0s] Welcome to lesson eight point nine on multimodal RAG'), showing the timestamped chunks retrieval will index.

## 9 - Production: routing and a content-hash cache

The by-hand pipeline works on ten documents; production runs on ten million, where the VLM description that felt free becomes the dominant cost and the 1-5s latency spike. This step is the economics: route each modality to its cheapest sufficient extractor, and cache so you never describe the same content twice.

In [ ]:
# PRODUCTION - modality-aware routing + a content-hash embedding cache
import hashlib

def route(modality):
    # send each modality to its cheapest sufficient extractor; VLM only for images
    return {"text": "chunk+embed (ms, cheap)",
            "table": "extract->markdown (cheap)",
            "image": "VLM describe (1-5s, dominant cost)",
            "audio": "whisper transcribe",
            "video": "asr + frames + ocr"}[modality]

class EmbeddingCache:
    def __init__(self): self.store = {}
    def get_or_embed(self, text, embed_fn):
        key = hashlib.sha256(text.encode()).hexdigest()
        if key in self.store:
            return self.store[key], True          # cache hit - no API call
        self.store[key] = embed_fn(text)
        return self.store[key], False

print("router(image) ->", route("image"))
cache = EmbeddingCache()
hits = 0
for t in ["revenue chart", "revenue chart", "refund policy", "revenue chart"]:
    _, hit = cache.get_or_embed(t, lambda s: [len(s)])
    hits += int(hit)
print(f"cache hits: {hits}/4 (repeats served from cache, no re-embed)")

# Output:
# router(image) -> VLM describe (1-5s, dominant cost)
# cache hits: 2/4 (repeats served from cache, no re-embed)

**Explanation**

Pure Python, no model calls - the two levers that make multimodal RAG affordable. `route` maps each modality to its cheapest sufficient extractor (VLM reserved for images), and `EmbeddingCache` hashes content so repeated text or charts are served from memory instead of re-embedded.

**How the code works - step by step**
- **`route`** - a dict mapping each modality to its handler, flagging image -> VLM as the 1-5s dominant cost.
- **`EmbeddingCache.get_or_embed`** - SHA-256-hashes the text; on a hit it returns the stored vector with no API call, otherwise it embeds once and stores.
- **demo** - routes an image, then runs four strings ('revenue chart' three times) through the cache and counts hits.

**In one line:** route to avoid the VLM, cache to avoid embedding anything twice.

**What you'll see:** Prints that routing an image yields 'VLM describe (1-5s, dominant cost)', then 'cache hits: 2/4' - the two repeated 'revenue chart' strings were served from cache without re-embedding.

## 10 - India multimodal RAG: bilingual OCR and field validation

Indian document processing hits problems the English demos never surface - Devanagari is far harder to OCR than Latin, ID cards mix Hindi and English with stamps and overlays. This step shows bilingual OCR with validation, and the DigiLocker shortcut that skips OCR entirely when the document is available.

> **Needs Google Cloud Vision credentials** (set GOOGLE_APPLICATION_CREDENTIALS) to run the OCR call live.

In [ ]:
# INDIA MULTIMODAL RAG - bilingual OCR + structured-field validation
# pip install google-cloud-vision
import re
from google.cloud import vision

def ocr_hindi_english(path):
    client = vision.ImageAnnotatorClient()
    with open(path, "rb") as f:
        image = vision.Image(content=f.read())
    ctx = vision.ImageContext(language_hints=["hi", "en"])   # bilingual ID cards
    resp = client.document_text_detection(image=image, image_context=ctx)   # dense/document text
    return resp.full_text_annotation.text

# validate extracted ID fields BEFORE trusting them
AADHAAR = re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b")
PAN = re.compile(r"\b[A-Z]{5}[0-9]{4}[A-Z]\b")

text = "Name: Asha   Aadhaar: 1234 5678 9012   PAN: ABCDE1234F"
print("aadhaar:", AADHAAR.search(text).group())
print("pan:    ", PAN.search(text).group())

# Best shortcut: DigiLocker API returns legally-valid structured XML for hundreds of millions of users -
# parse fields directly and skip OCR entirely when the document is available there.

# Output:
# aadhaar: 1234 5678 9012
# pan:     ABCDE1234F

**Explanation**

Two guardrails for real ID documents. `ocr_hindi_english` runs Google Cloud Vision with bilingual language hints for Devanagari + English, and a pair of regexes validate the extracted Aadhaar and PAN against their known formats before any field is trusted - the check that stops a misread digit from silently corrupting data.

**How the code works - step by step**
- **`ocr_hindi_english`** - reads the image bytes and runs `document_text_detection` with `language_hints=["hi","en"]` for dense bilingual card text.
- **`AADHAAR` / `PAN` regexes** - encode the exact formats (12 digits in 4-4-4 groups; 5 letters + 4 digits + 1 letter).
- **validation demo** - runs both patterns over a sample string and prints the matched Aadhaar and PAN.
- comment - when the document exists in DigiLocker, pull its legally-valid structured XML and skip OCR entirely.

**In one line:** OCR bilingual text, validate every field by regex, or skip OCR via DigiLocker.

**What you'll see:** Prints the validated fields extracted from the sample string: 'aadhaar: 1234 5678 9012' and 'pan: ABCDE1234F', showing the format check passing before the fields are trusted.

Across ten cells you turned text-only RAG into multimodal RAG by adding exactly one stage - get every modality into a text vector space - then toured the industrial versions of that move. The through-line: describe an image/table/audio/video to text and embed the text (cheap, one retriever finds everything), retrieve on descriptions but send the real image at generation when a query needs an exact value, and reach for pixel embeddings (CLIP/ColPali) only when layout itself carries meaning. Next up: Lesson 8.10 lets an agent pick the modality path at runtime, and 8.11 measures whether the right chart or table actually got retrieved.